# 02 - DuckDB Warehouse

This notebook creates a local DuckDB analytical warehouse for the Amsterdam Inside Airbnb dataset.

The purpose of this notebook is to load the cleaned listing master dataset into DuckDB and prepare SQL-based analytical tables for reporting, EDA, and future dashboarding.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.4


In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "amsterdam"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse"

WAREHOUSE_PATH.mkdir(parents=True, exist_ok=True)

DUCKDB_DATABASE_PATH = WAREHOUSE_PATH / "airbnb_market.duckdb"
LISTING_MASTER_FINAL_PATH = PROCESSED_DATA_PATH / "listing_master_final.parquet"

print("DuckDB database path:", DUCKDB_DATABASE_PATH)
print("Listing master path exists:", LISTING_MASTER_FINAL_PATH.exists())
print("Listing master path:", LISTING_MASTER_FINAL_PATH)

DuckDB database path: ..\warehouse\airbnb_market.duckdb
Listing master path exists: True
Listing master path: ..\data\processed\amsterdam\listing_master_final.parquet


In [3]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

print("Connected to DuckDB successfully.")

Connected to DuckDB successfully.


In [4]:
conn.execute("""
CREATE OR REPLACE TABLE listing_master_final AS
SELECT *
FROM read_parquet(?)
""", [str(LISTING_MASTER_FINAL_PATH)])

print("listing_master_final table created successfully in DuckDB.")

listing_master_final table created successfully in DuckDB.


In [5]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,listing_master_final


In [6]:
conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT listing_id) AS unique_listing_ids
FROM listing_master_final
""").fetchdf()

,row_count,unique_listing_ids
0,10465,10465


In [7]:
conn.execute("""
SELECT 
    room_type,
    COUNT(*) AS listing_count,
    ROUND(AVG(price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_price,
    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy
FROM listing_master_final
GROUP BY room_type
ORDER BY listing_count DESC
""").fetchdf()

,room_type,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy
0,Entire home/apt,8489,394.62,331.0,0.2133,0.7867
1,Private room,1929,203.67,171.0,0.4612,0.5388
2,Hotel room,26,249.85,227.5,0.7564,0.2436
3,Shared room,21,80.10,52.0,0.7769,0.2231


In [8]:
conn.close()

print("DuckDB connection closed.")

DuckDB connection closed.


## 2. Star Schema Design

This section creates a simple dimensional model in DuckDB using the final listing master dataset.

The model includes:
- dim_listing
- dim_host
- dim_neighbourhood
- fact_listing_market

The purpose is to separate descriptive attributes from measurable market metrics and support SQL-based analytical queries.

In [9]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

columns_df = conn.execute("""
DESCRIBE listing_master_final
""").fetchdf()

columns_df

,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,listing_name,VARCHAR,YES,None,None,None
2,host_id,BIGINT,YES,None,None,None
3,host_profile_id,BIGINT,YES,None,None,None
4,host_name,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
96,neighbourhood_total_reviews,BIGINT,YES,None,None,None
97,neighbourhood_avg_reviews_per_listing,DOUBLE,YES,None,None,None
98,neighbourhood_total_revenue_proxy,DOUBLE,YES,None,None,None
99,neighbourhood_avg_revenue_proxy,DOUBLE,YES,None,None,None


### Dimension Table: dim_listing

The listing dimension stores descriptive listing and property-level attributes.  
Each row represents one Airbnb listing.

In [10]:
conn.execute("""
CREATE OR REPLACE TABLE dim_listing AS
SELECT
    ROW_NUMBER() OVER (ORDER BY listing_id) AS listing_key,
    listing_id,
    listing_name,
    room_type,
    property_type,
    accommodates,
    bathrooms,
    bathrooms_text,
    bedrooms,
    beds,
    amenities_count,
    instant_bookable,
    latitude,
    longitude,
    city,
    country,
    snapshot_date
FROM listing_master_final
""")

conn.execute("""
SELECT COUNT(*) AS row_count, COUNT(DISTINCT listing_id) AS unique_listing_ids
FROM dim_listing
""").fetchdf()

,row_count,unique_listing_ids
0,10465,10465


### Dimension Table: dim_host

The host dimension stores host-level descriptive attributes.  
Each row represents one unique host.

In [20]:
conn.execute("""
CREATE OR REPLACE TABLE dim_host AS
WITH known_hosts AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY host_id) AS host_key,
        host_id,
        ANY_VALUE(host_profile_id) AS host_profile_id,
        ANY_VALUE(host_name) AS host_name,
        ANY_VALUE(host_since) AS host_since,
        ANY_VALUE(host_is_superhost) AS host_is_superhost,
        ANY_VALUE(host_response_time) AS host_response_time,
        ANY_VALUE(host_response_rate) AS host_response_rate,
        ANY_VALUE(host_acceptance_rate) AS host_acceptance_rate,
        ANY_VALUE(host_listings_count) AS host_listings_count,
        ANY_VALUE(host_total_listings_count) AS host_total_listings_count,
        ANY_VALUE(host_identity_verified) AS host_identity_verified,
        ANY_VALUE(calculated_host_listings_count) AS calculated_host_listings_count,
        ANY_VALUE(host_portfolio_segment) AS host_portfolio_segment
    FROM listing_master_final
    WHERE host_id IS NOT NULL
    GROUP BY host_id
),

unknown_host AS (
    SELECT
        -1 AS host_key,
        NULL AS host_id,
        NULL AS host_profile_id,
        'Unknown Host' AS host_name,
        NULL AS host_since,
        NULL AS host_is_superhost,
        NULL AS host_response_time,
        NULL AS host_response_rate,
        NULL AS host_acceptance_rate,
        NULL AS host_listings_count,
        NULL AS host_total_listings_count,
        NULL AS host_identity_verified,
        NULL AS calculated_host_listings_count,
        'Unknown Host' AS host_portfolio_segment
)

SELECT * FROM unknown_host
UNION ALL
SELECT * FROM known_hosts
""")

conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT host_id) AS unique_host_ids,
    SUM(CASE WHEN host_key = -1 THEN 1 ELSE 0 END) AS unknown_host_records
FROM dim_host
""").fetchdf()

,row_count,unique_host_ids,unknown_host_records
0,9105,9104,1.0


### Dimension Table: dim_neighbourhood

The neighbourhood dimension stores geographic and neighbourhood-level market context.  
Each row represents one Amsterdam neighbourhood.

In [21]:
conn.execute("""
CREATE OR REPLACE TABLE dim_neighbourhood AS
SELECT
    ROW_NUMBER() OVER (ORDER BY neighbourhood) AS neighbourhood_key,
    neighbourhood,
    ANY_VALUE(neighbourhood_group) AS neighbourhood_group,
    ANY_VALUE(city) AS city,
    ANY_VALUE(country) AS country,
    ANY_VALUE(neighbourhood_listing_count) AS neighbourhood_listing_count,
    ANY_VALUE(neighbourhood_unique_hosts) AS neighbourhood_unique_hosts,
    ANY_VALUE(neighbourhood_median_price) AS neighbourhood_median_price,
    ANY_VALUE(neighbourhood_avg_price) AS neighbourhood_avg_price,
    ANY_VALUE(neighbourhood_avg_availability_rate) AS neighbourhood_avg_availability_rate,
    ANY_VALUE(neighbourhood_avg_occupancy_proxy) AS neighbourhood_avg_occupancy_proxy,
    ANY_VALUE(neighbourhood_avg_review_score) AS neighbourhood_avg_review_score,
    ANY_VALUE(neighbourhood_dominant_room_type) AS neighbourhood_dominant_room_type
FROM listing_master_final
WHERE neighbourhood IS NOT NULL
GROUP BY neighbourhood
""")

conn.execute("""
SELECT COUNT(*) AS row_count, COUNT(DISTINCT neighbourhood) AS unique_neighbourhoods
FROM dim_neighbourhood
""").fetchdf()

,row_count,unique_neighbourhoods
0,22,22


### Fact Table: fact_listing_market

The fact table stores listing-level measurable market metrics.

It connects to:
- dim_listing through listing_key
- dim_host through host_key
- dim_neighbourhood through neighbourhood_key

In [22]:
conn.execute("""
CREATE OR REPLACE TABLE fact_listing_market AS
SELECT
    dl.listing_key,
    COALESCE(dh.host_key, -1) AS host_key,
    dn.neighbourhood_key,

    lm.listing_id,
    lm.host_id,
    lm.neighbourhood,

    lm.price_numeric,
    lm.minimum_nights,
    lm.number_of_reviews,
    lm.reviews_per_month,
    lm.availability_365,

    lm.calendar_days,
    lm.available_days,
    lm.unavailable_days,
    lm.availability_rate,
    lm.occupancy_proxy,
    lm.weekend_availability_rate,
    lm.weekday_availability_rate,

    lm.estimated_revenue_proxy,
    lm.listing_price_available_flag,

    lm.detailed_review_count,
    lm.unique_reviewer_count,
    lm.detailed_reviews_last_365d,
    lm.avg_reviews_per_year,
    lm.has_reviews,

    lm.review_scores_rating,
    lm.review_scores_cleanliness,
    lm.review_scores_location,
    lm.review_scores_value
FROM listing_master_final lm
LEFT JOIN dim_listing dl
    ON lm.listing_id = dl.listing_id
LEFT JOIN dim_host dh
    ON lm.host_id = dh.host_id
LEFT JOIN dim_neighbourhood dn
    ON lm.neighbourhood = dn.neighbourhood
""")

conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT listing_id) AS unique_listing_ids,
    COUNT(*) - COUNT(listing_key) AS missing_listing_keys,
    SUM(CASE WHEN host_key IS NULL THEN 1 ELSE 0 END) AS missing_host_keys,
    COUNT(*) - COUNT(neighbourhood_key) AS missing_neighbourhood_keys,
    SUM(CASE WHEN host_key = -1 THEN 1 ELSE 0 END) AS unknown_host_fact_rows
FROM fact_listing_market
""").fetchdf()

,row_count,unique_listing_ids,missing_listing_keys,missing_host_keys,missing_neighbourhood_keys,unknown_host_fact_rows
0,10465,10465,0,0.0,0,96.0


In [23]:
conn.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_host
1,dim_listing
2,dim_neighbourhood
3,fact_listing_market
4,listing_master_final


In [24]:
star_schema_validation = conn.execute("""
SELECT 'dim_listing' AS table_name, COUNT(*) AS row_count FROM dim_listing
UNION ALL
SELECT 'dim_host' AS table_name, COUNT(*) AS row_count FROM dim_host
UNION ALL
SELECT 'dim_neighbourhood' AS table_name, COUNT(*) AS row_count FROM dim_neighbourhood
UNION ALL
SELECT 'fact_listing_market' AS table_name, COUNT(*) AS row_count FROM fact_listing_market
""").fetchdf()

star_schema_validation

,table_name,row_count
0,dim_listing,10465
1,dim_host,9105
2,dim_neighbourhood,22
3,fact_listing_market,10465


In [25]:
REPORTS_PATH = PROJECT_ROOT / "reports" / "data_quality"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

star_schema_validation_output_path = REPORTS_PATH / "amsterdam_star_schema_validation.csv"

star_schema_validation.to_csv(star_schema_validation_output_path, index=False)

print(f"Star schema validation saved to: {star_schema_validation_output_path}")

Star schema validation saved to: ..\reports\data_quality\amsterdam_star_schema_validation.csv


In [26]:
conn.execute("""
SELECT
    dn.neighbourhood,
    dl.room_type,
    COUNT(*) AS listing_count,
    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,
    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,
    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy
FROM fact_listing_market f
LEFT JOIN dim_listing dl
    ON f.listing_key = dl.listing_key
LEFT JOIN dim_neighbourhood dn
    ON f.neighbourhood_key = dn.neighbourhood_key
GROUP BY
    dn.neighbourhood,
    dl.room_type
ORDER BY
    total_revenue_proxy DESC
LIMIT 10
""").fetchdf()

,neighbourhood,room_type,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,total_revenue_proxy
0,De Baarsjes - Oud-West,Entire home/apt,1633,401.29,357.0,0.1922,0.8078,83509991.0
1,De Pijp - Rivierenbuurt,Entire home/apt,1057,423.24,361.0,0.2268,0.7732,48674461.0
2,Centrum-West,Entire home/apt,766,518.39,400.0,0.2647,0.7353,47399839.0
3,Centrum-Oost,Entire home/apt,614,445.48,383.0,0.2976,0.7024,34440350.0
4,Zuid,Entire home/apt,599,459.20,360.0,0.2540,0.7460,32455208.0
5,Westerpark,Entire home/apt,652,354.33,303.5,0.1890,0.8110,29228261.0
6,Oud-Oost,Entire home/apt,571,345.81,304.0,0.1925,0.8075,23498888.0
7,Oud-Noord,Entire home/apt,403,392.91,285.0,0.1828,0.8172,17648085.0
8,Centrum-West,Private room,447,256.67,206.0,0.4610,0.5390,17390977.0
9,Bos en Lommer,Entire home/apt,507,300.40,281.0,0.1557,0.8443,16878453.0


### Star Schema Observations

A simple DuckDB star schema was created using the final listing master dataset.

The model includes listing, host, and neighbourhood dimension tables, plus a listing-level market fact table.  
The fact table stores measurable metrics such as price, availability, occupancy proxy, review activity, review scores, and estimated revenue proxy.

This structure supports SQL-based analytical queries and separates descriptive attributes from measurable business metrics.

### Unknown Host Handling

During star schema validation, 96 fact rows did not have a matching host key because their host_id values were missing in the source listing data.

To preserve referential consistency, an Unknown Host record was added to dim_host with host_key = -1.  
Fact rows with missing host information are assigned to this unknown host key.

This approach preserves all listing records while clearly documenting incomplete host metadata.

## 3. Analytical SQL Summary Tables

This section creates business-ready analytical summary tables using the DuckDB star schema.

The summary tables are designed for exploratory analysis, dashboarding, reporting, and business interpretation.

Created tables:
- market_overview
- room_type_summary
- neighbourhood_summary
- host_portfolio_summary
- review_score_summary

In [27]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

print("DuckDB connection is ready.")

DuckDB connection is ready.


In [28]:
ANALYTICS_REPORTS_PATH = PROJECT_ROOT / "reports" / "analytics_outputs"
ANALYTICS_REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("Analytics reports folder:", ANALYTICS_REPORTS_PATH)

Analytics reports folder: ..\reports\analytics_outputs


### Market Overview Table

The market_overview table provides a one-row executive summary of the Amsterdam Airbnb market.

In [29]:
conn.execute("""
CREATE OR REPLACE TABLE market_overview AS
SELECT
    COUNT(*) AS total_listings,
    COUNT(DISTINCT host_id) AS total_known_hosts,
    COUNT(DISTINCT neighbourhood) AS total_neighbourhoods,

    ROUND(AVG(price_numeric), 2) AS avg_listing_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_listing_price,

    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy,

    SUM(CASE WHEN has_reviews = TRUE THEN 1 ELSE 0 END) AS listings_with_reviews,
    SUM(CASE WHEN has_reviews = FALSE THEN 1 ELSE 0 END) AS listings_without_reviews,

    SUM(detailed_review_count) AS total_detailed_reviews,
    ROUND(AVG(review_scores_rating), 2) AS avg_review_score,

    ROUND(SUM(estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market
""")

market_overview_df = conn.execute("""
SELECT * FROM market_overview
""").fetchdf()

market_overview_df

,total_listings,total_known_hosts,total_neighbourhoods,avg_listing_price,median_listing_price,avg_availability_rate,avg_occupancy_proxy,listings_with_reviews,listings_without_reviews,total_detailed_reviews,avg_review_score,total_revenue_proxy,avg_revenue_proxy,listings_missing_price
0,10465,9104,22,344.24,287.0,0.2615,0.7385,9432.0,1033.0,545162.0,4.85,464174714.0,71731.53,3994.0


In [30]:
market_overview_df.to_csv(
    ANALYTICS_REPORTS_PATH / "market_overview.csv",
    index=False
)

print("market_overview.csv saved.")

market_overview.csv saved.


### Room Type Summary Table

The room_type_summary table compares listing supply, pricing, availability, occupancy proxy, reviews, and revenue proxy by room type.

In [31]:
conn.execute("""
CREATE OR REPLACE TABLE room_type_summary AS
SELECT
    dl.room_type,

    COUNT(*) AS listing_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS market_share_percentage,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN f.price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market f
LEFT JOIN dim_listing dl
    ON f.listing_key = dl.listing_key
GROUP BY
    dl.room_type
ORDER BY
    listing_count DESC
""")

room_type_summary_df = conn.execute("""
SELECT * FROM room_type_summary
""").fetchdf()

room_type_summary_df

,room_type,listing_count,market_share_percentage,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,avg_review_score,total_reviews,total_revenue_proxy,avg_revenue_proxy,listings_missing_price
0,Entire home/apt,8489,81.12,394.62,331.0,0.2133,0.7867,4.87,221489.0,407899491.0,85495.60,3718.0
1,Private room,1929,18.43,203.67,171.0,0.4612,0.5388,4.77,311510.0,55409327.0,33520.46,276.0
2,Hotel room,26,0.25,249.85,227.5,0.7564,0.2436,4.78,6169.0,623504.0,23980.92,0.0
3,Shared room,21,0.20,80.10,52.0,0.7769,0.2231,4.52,5994.0,242392.0,11542.48,0.0


In [32]:
room_type_summary_df.to_csv(
    ANALYTICS_REPORTS_PATH / "room_type_summary.csv",
    index=False
)

print("room_type_summary.csv saved.")

room_type_summary.csv saved.


### Neighbourhood Summary Table

The neighbourhood_summary table compares listing supply, pricing, availability, occupancy proxy, review activity, and revenue proxy across Amsterdam neighbourhoods.

In [33]:
conn.execute("""
CREATE OR REPLACE TABLE neighbourhood_summary AS
SELECT
    dn.neighbourhood,

    COUNT(*) AS listing_count,
    COUNT(DISTINCT f.host_id) AS known_host_count,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,
    ROUND(AVG(f.detailed_review_count), 2) AS avg_reviews_per_listing,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN f.price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market f
LEFT JOIN dim_neighbourhood dn
    ON f.neighbourhood_key = dn.neighbourhood_key
GROUP BY
    dn.neighbourhood
ORDER BY
    total_revenue_proxy DESC
""")

neighbourhood_summary_df = conn.execute("""
SELECT * FROM neighbourhood_summary
""").fetchdf()

neighbourhood_summary_df.head(10)

,neighbourhood,listing_count,known_host_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,avg_review_score,total_reviews,avg_reviews_per_listing,total_revenue_proxy,avg_revenue_proxy,listings_missing_price
0,De Baarsjes - Oud-West,1825,1710,364.68,328.0,0.2261,0.7739,4.87,69085.0,37.85,88390081.0,82376.59,752.0
1,Centrum-West,1217,923,398.36,290.0,0.3376,0.6624,4.82,116784.0,95.96,65092219.0,74221.46,340.0
2,De Pijp - Rivierenbuurt,1201,1113,380.78,333.5,0.2522,0.7478,4.86,47109.0,39.22,53207727.0,75794.48,499.0
3,Centrum-Oost,933,708,350.41,282.5,0.3600,0.6400,4.81,85025.0,91.13,44237061.0,65246.40,255.0
4,Zuid,709,649,401.85,327.0,0.2984,0.7016,4.88,28645.0,40.40,35151047.0,78113.44,259.0
5,Westerpark,720,680,332.57,286.0,0.2152,0.7848,4.87,27910.0,38.76,30809751.0,76073.46,315.0
6,Oud-Oost,670,598,308.24,276.0,0.2418,0.7582,4.86,25575.0,38.17,25874801.0,66687.63,282.0
7,Oud-Noord,479,428,357.02,266.0,0.2248,0.7752,4.85,25295.0,52.81,19369930.0,68444.98,196.0
8,Bos en Lommer,546,527,281.79,265.0,0.1675,0.8325,4.87,17223.0,31.54,17675443.0,68776.04,289.0
9,Oostelijk Havengebied - Indische Buurt,411,391,290.24,262.0,0.2123,0.7877,4.86,16186.0,39.38,14748957.0,63848.30,180.0


In [34]:
neighbourhood_summary_df.to_csv(
    ANALYTICS_REPORTS_PATH / "neighbourhood_summary.csv",
    index=False
)

print("neighbourhood_summary.csv saved.")

neighbourhood_summary.csv saved.


### Host Portfolio Summary Table

The host_portfolio_summary table compares market behavior across host portfolio segments.

In [35]:
conn.execute("""
CREATE OR REPLACE TABLE host_portfolio_summary AS
SELECT
    dh.host_portfolio_segment,

    COUNT(*) AS listing_count,
    COUNT(DISTINCT f.host_id) AS known_host_count,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy
FROM fact_listing_market f
LEFT JOIN dim_host dh
    ON f.host_key = dh.host_key
GROUP BY
    dh.host_portfolio_segment
ORDER BY
    listing_count DESC
""")

host_portfolio_summary_df = conn.execute("""
SELECT * FROM host_portfolio_summary
""").fetchdf()

host_portfolio_summary_df

,host_portfolio_segment,listing_count,known_host_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,avg_review_score,total_reviews,total_revenue_proxy,avg_revenue_proxy
0,Single-listing host,8509,8509,371.11,318.0,0.2085,0.7915,4.88,334320.0,396232619.0,82359.72
1,Small portfolio host,1309,537,274.87,202.0,0.4396,0.5604,4.76,161660.0,47775711.0,45113.99
2,Medium portfolio host,483,55,276.40,209.0,0.5640,0.4360,4.65,34771.0,17365435.0,39377.40
3,Unknown Host,96,0,177.53,141.0,0.7595,0.2405,NaN,9048.0,1314652.0,13985.66
4,Large portfolio host,68,3,189.39,151.0,0.6113,0.3887,4.54,5363.0,1486297.0,22519.65


In [36]:
host_portfolio_summary_df.to_csv(
    ANALYTICS_REPORTS_PATH / "host_portfolio_summary.csv",
    index=False
)

print("host_portfolio_summary.csv saved.")

host_portfolio_summary.csv saved.


### Review Score Summary Table

The review_score_summary table groups listings into review score bands and compares pricing, availability, occupancy proxy, and review activity.

In [37]:
conn.execute("""
CREATE OR REPLACE TABLE review_score_summary AS
SELECT
    CASE
        WHEN review_scores_rating IS NULL THEN 'No rating'
        WHEN review_scores_rating >= 4.8 THEN 'Excellent'
        WHEN review_scores_rating >= 4.5 THEN 'Very good'
        WHEN review_scores_rating >= 4.0 THEN 'Good'
        ELSE 'Below 4.0'
    END AS review_score_band,

    COUNT(*) AS listing_count,

    ROUND(AVG(price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_price,

    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy,

    SUM(detailed_review_count) AS total_reviews,
    ROUND(AVG(detailed_review_count), 2) AS avg_reviews_per_listing,

    ROUND(SUM(estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(estimated_revenue_proxy), 2) AS avg_revenue_proxy
FROM fact_listing_market
GROUP BY
    review_score_band
ORDER BY
    listing_count DESC
""")

review_score_summary_df = conn.execute("""
SELECT * FROM review_score_summary
""").fetchdf()

review_score_summary_df

,review_score_band,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,total_reviews,avg_reviews_per_listing,total_revenue_proxy,avg_revenue_proxy
0,Excellent,7047,356.29,309.0,0.2211,0.7789,345884.0,49.08,336075338.0,79960.82
1,Very good,1794,292.38,241.0,0.3018,0.6982,171054.0,95.35,67916253.0,57899.62
2,No rating,1119,389.79,286.0,0.3778,0.6222,9048.0,8.09,45524688.0,62277.27
3,Good,434,250.67,204.0,0.4136,0.5864,18741.0,43.18,12897782.0,41206.97
4,Below 4.0,71,465.57,212.0,0.4886,0.5114,435.0,6.13,1760653.0,34522.61


In [38]:
review_score_summary_df.to_csv(
    ANALYTICS_REPORTS_PATH / "review_score_summary.csv",
    index=False
)

print("review_score_summary.csv saved.")

review_score_summary.csv saved.


In [39]:
conn.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_host
1,dim_listing
2,dim_neighbourhood
3,fact_listing_market
4,host_portfolio_summary
5,listing_master_final
6,market_overview
7,neighbourhood_summary
8,review_score_summary
9,room_type_summary


In [40]:
analytics_tables_validation = conn.execute("""
SELECT 'market_overview' AS table_name, COUNT(*) AS row_count FROM market_overview
UNION ALL
SELECT 'room_type_summary' AS table_name, COUNT(*) AS row_count FROM room_type_summary
UNION ALL
SELECT 'neighbourhood_summary' AS table_name, COUNT(*) AS row_count FROM neighbourhood_summary
UNION ALL
SELECT 'host_portfolio_summary' AS table_name, COUNT(*) AS row_count FROM host_portfolio_summary
UNION ALL
SELECT 'review_score_summary' AS table_name, COUNT(*) AS row_count FROM review_score_summary
""").fetchdf()

analytics_tables_validation

,table_name,row_count
0,market_overview,1
1,room_type_summary,4
2,neighbourhood_summary,22
3,host_portfolio_summary,5
4,review_score_summary,5


In [41]:
analytics_tables_validation.to_csv(
    ANALYTICS_REPORTS_PATH / "analytics_tables_validation.csv",
    index=False
)

print("analytics_tables_validation.csv saved.")

analytics_tables_validation.csv saved.


### Analytical SQL Summary Observations

Business-ready analytical summary tables were created from the DuckDB star schema.

These summary tables provide market-level, room-type-level, neighbourhood-level, host-segment-level, and review-score-level views of the Amsterdam Airbnb market.

The outputs can be used for exploratory data analysis, dashboarding, final report tables, and business recommendations.

## 4. Export Analytical SQL Queries

This section saves key analytical SQL queries as standalone `.sql` files.

Keeping SQL queries outside the notebook improves project readability, reviewability, and reproducibility.  
These queries can be reused for reporting, dashboarding, business analysis, and validation.

In [42]:
SQL_ANALYSIS_PATH = PROJECT_ROOT / "sql" / "analysis_queries"
SQL_ANALYSIS_PATH.mkdir(parents=True, exist_ok=True)

print("SQL analysis query folder:", SQL_ANALYSIS_PATH)

SQL analysis query folder: ..\sql\analysis_queries


In [43]:
sql_queries = {
    "01_market_overview.sql": """
-- Market Overview
-- Purpose: Provides a one-row executive summary of the Amsterdam Airbnb market.

SELECT
    COUNT(*) AS total_listings,
    COUNT(DISTINCT host_id) AS total_known_hosts,
    COUNT(DISTINCT neighbourhood) AS total_neighbourhoods,

    ROUND(AVG(price_numeric), 2) AS avg_listing_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_listing_price,

    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy,

    SUM(CASE WHEN has_reviews = TRUE THEN 1 ELSE 0 END) AS listings_with_reviews,
    SUM(CASE WHEN has_reviews = FALSE THEN 1 ELSE 0 END) AS listings_without_reviews,

    SUM(detailed_review_count) AS total_detailed_reviews,
    ROUND(AVG(review_scores_rating), 2) AS avg_review_score,

    ROUND(SUM(estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market;
""",

    "02_room_type_summary.sql": """
-- Room Type Summary
-- Purpose: Compares market supply, pricing, availability, occupancy proxy, reviews, and revenue proxy by room type.

SELECT
    dl.room_type,

    COUNT(*) AS listing_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS market_share_percentage,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN f.price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market f
LEFT JOIN dim_listing dl
    ON f.listing_key = dl.listing_key
GROUP BY
    dl.room_type
ORDER BY
    listing_count DESC;
""",

    "03_neighbourhood_summary.sql": """
-- Neighbourhood Summary
-- Purpose: Compares supply, pricing, availability, occupancy proxy, review activity, and revenue proxy by neighbourhood.

SELECT
    dn.neighbourhood,

    COUNT(*) AS listing_count,
    COUNT(DISTINCT f.host_id) AS known_host_count,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,
    ROUND(AVG(f.detailed_review_count), 2) AS avg_reviews_per_listing,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy,

    SUM(CASE WHEN f.price_numeric IS NULL THEN 1 ELSE 0 END) AS listings_missing_price
FROM fact_listing_market f
LEFT JOIN dim_neighbourhood dn
    ON f.neighbourhood_key = dn.neighbourhood_key
GROUP BY
    dn.neighbourhood
ORDER BY
    total_revenue_proxy DESC;
""",

    "04_host_portfolio_summary.sql": """
-- Host Portfolio Summary
-- Purpose: Compares listing behavior across host portfolio segments.

SELECT
    dh.host_portfolio_segment,

    COUNT(*) AS listing_count,
    COUNT(DISTINCT f.host_id) AS known_host_count,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(AVG(f.review_scores_rating), 2) AS avg_review_score,
    SUM(f.detailed_review_count) AS total_reviews,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(f.estimated_revenue_proxy), 2) AS avg_revenue_proxy
FROM fact_listing_market f
LEFT JOIN dim_host dh
    ON f.host_key = dh.host_key
GROUP BY
    dh.host_portfolio_segment
ORDER BY
    listing_count DESC;
""",

    "05_review_score_summary.sql": """
-- Review Score Summary
-- Purpose: Groups listings by review score band and compares pricing, availability, occupancy proxy, and review activity.

SELECT
    CASE
        WHEN review_scores_rating IS NULL THEN 'No rating'
        WHEN review_scores_rating >= 4.8 THEN 'Excellent'
        WHEN review_scores_rating >= 4.5 THEN 'Very good'
        WHEN review_scores_rating >= 4.0 THEN 'Good'
        ELSE 'Below 4.0'
    END AS review_score_band,

    COUNT(*) AS listing_count,

    ROUND(AVG(price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_price,

    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy,

    SUM(detailed_review_count) AS total_reviews,
    ROUND(AVG(detailed_review_count), 2) AS avg_reviews_per_listing,

    ROUND(SUM(estimated_revenue_proxy), 2) AS total_revenue_proxy,
    ROUND(AVG(estimated_revenue_proxy), 2) AS avg_revenue_proxy
FROM fact_listing_market
GROUP BY
    review_score_band
ORDER BY
    listing_count DESC;
""",

    "06_top_neighbourhood_room_type_revenue.sql": """
-- Top Neighbourhood and Room Type Revenue Proxy
-- Purpose: Identifies neighbourhood-room type combinations with the highest estimated revenue proxy.

SELECT
    dn.neighbourhood,
    dl.room_type,
    COUNT(*) AS listing_count,

    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,

    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,

    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy
FROM fact_listing_market f
LEFT JOIN dim_listing dl
    ON f.listing_key = dl.listing_key
LEFT JOIN dim_neighbourhood dn
    ON f.neighbourhood_key = dn.neighbourhood_key
GROUP BY
    dn.neighbourhood,
    dl.room_type
ORDER BY
    total_revenue_proxy DESC
LIMIT 20;
"""
}

for file_name, query in sql_queries.items():
    file_path = SQL_ANALYSIS_PATH / file_name
    file_path.write_text(query.strip(), encoding="utf-8")

print(f"{len(sql_queries)} SQL query files created successfully.")

6 SQL query files created successfully.


In [44]:
created_sql_files = sorted(SQL_ANALYSIS_PATH.glob("*.sql"))

for file_path in created_sql_files:
    print(file_path.name)

01_market_overview.sql
02_room_type_summary.sql
03_neighbourhood_summary.sql
04_host_portfolio_summary.sql
05_review_score_summary.sql
06_top_neighbourhood_room_type_revenue.sql


In [45]:
test_query_path = SQL_ANALYSIS_PATH / "02_room_type_summary.sql"

test_query = test_query_path.read_text(encoding="utf-8")

conn.execute(test_query).fetchdf()

,room_type,listing_count,market_share_percentage,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,avg_review_score,total_reviews,total_revenue_proxy,avg_revenue_proxy,listings_missing_price
0,Entire home/apt,8489,81.12,394.62,331.0,0.2133,0.7867,4.87,221489.0,407899491.0,85495.60,3718.0
1,Private room,1929,18.43,203.67,171.0,0.4612,0.5388,4.77,311510.0,55409327.0,33520.46,276.0
2,Hotel room,26,0.25,249.85,227.5,0.7564,0.2436,4.78,6169.0,623504.0,23980.92,0.0
3,Shared room,21,0.20,80.10,52.0,0.7769,0.2231,4.52,5994.0,242392.0,11542.48,0.0


In [46]:
sql_readme_content = """
# SQL Analysis Queries

This folder contains standalone SQL queries used for business analysis against the DuckDB star schema.

## Query Files

1. `01_market_overview.sql`  
   Executive-level market summary.

2. `02_room_type_summary.sql`  
   Room type comparison by supply, price, availability, review activity, and revenue proxy.

3. `03_neighbourhood_summary.sql`  
   Neighbourhood-level market comparison.

4. `04_host_portfolio_summary.sql`  
   Host portfolio segment comparison.

5. `05_review_score_summary.sql`  
   Review score band analysis.

6. `06_top_neighbourhood_room_type_revenue.sql`  
   Top neighbourhood and room type combinations by estimated revenue proxy.

## Important Note

`estimated_revenue_proxy` is not actual Airbnb revenue.  
It is a proxy calculated using listing-level price and unavailable calendar days.  
Unavailable days may include both booked nights and host-blocked dates.
"""

(SQL_ANALYSIS_PATH / "README.md").write_text(sql_readme_content.strip(), encoding="utf-8")

print("SQL analysis README.md created.")

SQL analysis README.md created.


### SQL Query Export Observations

Key analytical SQL queries were saved as standalone `.sql` files under `sql/analysis_queries`.

These files improve project reproducibility and make the SQL analysis easier to review outside the notebook.  
The exported queries support market overview analysis, room type comparison, neighbourhood comparison, host portfolio analysis, review score analysis, and top revenue proxy combinations.